Install the following

In [ ]:
!pip install python-doctr
!pip install tf2onnx
!pip install matplotlib

Image Loading and Data Extraction

In [ ]:
import os
import tempfile
import json
from io import StringIO
import datetime
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from google.colab import files

# Upload your image
uploaded = files.upload()

# Get the uploaded image filename (assuming a single image)
image_filename = list(uploaded.keys())[0]

# Load the DocTR OCR model
predictor = ocr_predictor(pretrained=True)

def get_unique_filename():
    """Generates a unique filename with a timestamp."""
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    return f"extracted_{timestamp}.json"

def append_to_json(data, filename):
    """Appends data to a JSON file.

    Args:
        data: The data to be appended (list or dictionary).
        filename: The name of the JSON file.
    """
    try:
        # Create a folder named "Extracted" if it doesn't exist
        if not os.path.exists("Extracted"):
            os.makedirs("Extracted")

        json_path = os.path.join("Extracted", filename)

        # Check if the file exists and create it if it doesn't
        if not os.path.exists(json_path):
            with open(json_path, 'w') as f:
                json.dump([], f, indent=4)  # Write an empty list initially

        # Load existing data
        with open(json_path, 'r+') as f:
            existing_data = json.load(f)
            existing_data.extend(data)  # Append new data
            f.seek(0)  # Move pointer to the beginning of the file
            json.dump(existing_data, f, indent=4)  # Dump data with indentation

        print(f"Extracted text successfully stored in: {json_path}")
    except (IOError, json.JSONDecodeError) as e:
        print(f"Error appending data to JSON: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Save the uploaded file to a temporary file
with tempfile.NamedTemporaryFile(delete=False) as temp_file:
    temp_file.write(uploaded[image_filename])
    temp_filename = temp_file.name

# Load the image from the temporary file
doc = DocumentFile.from_images(temp_filename)

# Use the OCR predictor to extract text
result = predictor(doc)

# List to store extracted text
extracted_text = []

# Iterate over pages in the Document object
for page in result.pages:
    if not page.blocks:
        continue  # Skip empty pages

    for block in page.blocks:
        if not block.lines:
            continue  # Skip empty blocks

        for line in block.lines:
            text_builder = StringIO()
            for word in line.words:
                if word.confidence >= 0.5:  # Only include words with confidence >= 50%
                    text_builder.write(word.value + " ")

            text = text_builder.getvalue().strip()  # Remove trailing space
            if text:
                extracted_text.append(text)
                print(text)

#result.show() to show the imag
# Generate a unique JSON filename
json_filename = get_unique_filename()

# Append extracted text to the JSON file
append_to_json(extracted_text, json_filename)


Data Sorting  

In [ ]:
import os
import json
import re


def extract_dates_and_codes(content):
    """
    Extract dates in DD/MM/YYYY and DD-MM-YYYY formats, and specific 10-character codes from a string content.
    """
    # Extract dates in DD/MM/YYYY and DD-MM-YYYY formats
    dates = re.findall(r'\b\d{2}[-/]\d{2}[-/]\d{4}\b', content)
    # Extract specific 10-character codes
    codes = re.findall(r'\b[A-Z]{5}\d{4}[A-Z]\b', content)
    return dates, codes


def extract_uppercase_strings(content):
    """
    Extract strings containing only uppercase letters (ignoring spaces),
    and remove words containing specific substrings and single-letter strings.
    Also remove strings of length 3 or less that don't contain any vowels, the string "HTA",
    and specific excluded strings.
    """
    # Remove spaces before searching
    content_no_spaces = content.replace(" ", "")

    # Extract uppercase strings
    uppercase_strings = re.findall(r'\b[A-Z]+\b', content_no_spaces)

    # Specific strings to exclude
    exclude_strings = [
        "HTA", "HRAERCDR", "HRRETTKRAR", "HRARCDR", "INCOMSIAXDEAKIMENT", "YYYY", "HRAERCPR", "HRGRCR", "HRARCOR", "ERTUR", "PAN", "UWHPYSGVJR", "TTHI","HTHINAME", "FATHER", "HRA"
    ]

    # List of substrings to exclude
    exclude_substrings = [
        "INCOME", "DEPART", "TAX", "OFINDIA", "GOVT", "SNAME", "SIGNATURE", "OFFINDIA", "XXXXXXXXXX", "PERMANENT", "ACCOUNTNUMBER", "OFBIRTH", "OFFBIRTH"
    ]

    # Filter out strings containing any of the exclude substrings, specific strings, with length 1, and specific short strings without vowels
    filtered_strings = [
        string for string in uppercase_strings
        if (not any(substring in string for substring in exclude_substrings)) and
           len(string) > 1 and
           (len(string) > 3 or any(vowel in string for vowel in 'AEIOU')) and
           string not in exclude_strings
    ]

    return filtered_strings


def process_json_file(file_path):
    """
    Process a JSON file to extract dates, codes, and uppercase strings.
    """
    with open(file_path, 'r') as json_file:
        data = json.load(json_file)
        content = json.dumps(data)  # Convert JSON data to string for processing
        dates, codes = extract_dates_and_codes(content)
        uppercase_strings = extract_uppercase_strings(content)
    return dates, codes, uppercase_strings


def scan_directory(directory):
    """
    Scan all JSON files in a directory to extract dates, codes, and uppercase strings.
    """
    extracted_data = []
    for filename in os.listdir(directory):
        if filename.endswith(".json"):
            file_path = os.path.join(directory, filename)
            dates, codes, uppercase_strings = process_json_file(file_path)

            # Remove duplicates from dates, codes, and uppercase_strings
            unique_dates = list(set(dates))
            unique_codes = list(set(codes))
            unique_uppercase_strings = list(set(uppercase_strings))

            extracted_data.append({
                "file_name": filename,  # Name of the JSON file
                "DOB": unique_dates,  # List of extracted dates (both DD-MM-YYYY and DD/MM/YYYY formats)
                "PAN": unique_codes,  # List of extracted codes
                "uppercase_strings": unique_uppercase_strings  # List of filtered uppercase strings (excluding single letters and specific short strings)
            })
    return extracted_data


def save_to_json(data, output_file):
    """
    Save extracted data to a JSON file.
    """
    with open(output_file, 'w') as json_file:
        json.dump(data, json_file, indent=4)


# Specify the directory containing the JSON files
directory_path = '/content/Extracted'

# Specify the output file name
output_file = 'extracted_data.json'

# Get the extracted information from all JSON files in the directory
all_extracted_data = scan_directory(directory_path)

# Save the extracted information to a JSON file
save_to_json(all_extracted_data, output_file)

print(f'Extracted data has been saved to {output_file}')


Extracted data has been saved to extracted_data.json


Data Analysis

In [ ]:
import json

def remove_empty_entries(file_path, rejected_log_path):
    # Read the JSON file
    with open(file_path, 'r') as f:
        data = json.load(f)

    # Lists to store cleaned data and rejected entries
    cleaned_data = []
    rejected_entries = []

    # Process each entry
    for entry in data:
        if entry.get("PAN") and entry.get("DOB") and entry.get("uppercase_strings"):
            cleaned_data.append(entry)
        else:
            reason = []
            if not entry.get("PAN"):
                reason.append("Unable to find a PAN number")
            if not entry.get("DOB"):
                reason.append("Unable to find a DOB")
            if not entry.get("uppercase_strings"):
                reason.append("Unable to find uppercase_strings")
            rejected_entries.append({
                "file_name": entry.get("file_name"),
                "Reason": "; ".join(reason)
            })

    # Write the cleaned data back to the same JSON file
    with open(file_path, 'w') as f:
        json.dump(cleaned_data, f, indent=4)

    # Write the rejected entries to the rejected log file
    with open(rejected_log_path, 'w') as f:
        json.dump(rejected_entries, f, indent=4)

# Example usage
file_path = '/content/extracted_data.json'
rejected_log_path = '/content/rejected.json'
remove_empty_entries(file_path, rejected_log_path)


Image Deletion

In [ ]:
import os

def delete_image_files(folder_path):
    # Define common image file extensions
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}

    # Iterate over all files in the folder
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        # Check if the file has one of the image extensions
        if os.path.isfile(file_path) and os.path.splitext(filename)[1].lower() in image_extensions:
            os.remove(file_path)
            print(f'Deleted: {file_path}')

# Example usage
folder_path = '/content/'
delete_image_files(folder_path)
